# Build a brand campaign from a product brief

Turn one fictional product brief into a headline, tagline, email, social ad, and visual direction. You will use OpenAI to write the copy, Agent Learning Kit to check the response format, and Pillow to compose a PNG preview from the approved fields. The preview is a layout, not generated product photography. Nothing is published or sent to customers.

## Install and configure

Start with a fresh Python 3.12 kernel. Set `OPENAI_API_KEY` in your environment or enter it at the hidden prompt. The local evaluator needs no Future AGI credentials.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7", "openai==2.54.0", "gradio==6.32.0", "pillow>=10.1,<12"], check=True);

In [ ]:
import os
from getpass import getpass

def require_secret(name):
    if os.environ.get(name):
        return
    value = getpass(f"{name} (hidden): ")
    if not value.strip():
        raise ValueError(f"{name} is required for this step")
    os.environ[name] = value.strip()

require_secret("OPENAI_API_KEY")

## Write the brief and specify the deliverables

Use product facts that you can substantiate. Separating the headline, copy, and colors into fields lets you use the same output in an email draft and a social card.

In [ ]:
from typing import Annotated
from pydantic import BaseModel, Field, StringConstraints
from openai import OpenAI
from fi.evals import evaluate

Short = Annotated[str, StringConstraints(strip_whitespace=True, min_length=1)]
HexColor = Annotated[str, StringConstraints(pattern=r"^#[0-9a-fA-F]{6}$")]

class Brief(BaseModel):
    brand: Short
    product: Short
    audience: Short
    facts: Short
    objective: Short

class Campaign(BaseModel):
    headline: Short
    tagline: Short
    email_subject: Short
    email_body: Short
    social_ad: Short
    call_to_action: Short
    logo_concept: Short
    typography: Short
    palette: list[HexColor] = Field(min_length=3, max_length=3)

brief = Brief(brand="Ridgeline Outfitters", product="Switchback Daypack",
              audience="Weekend hikers who take short trails",
              facts="18-liter capacity, two side pockets, adjustable straps.",
              objective="Invite readers to explore the daypack.")

## Generate and validate the copy

The local `is_json` metric checks syntax. The schema checks fields and colors; neither can establish that a marketing claim is true.

In [ ]:
import json

client = OpenAI()

def generate(brief: Brief) -> Campaign:
    response = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0.3,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": (
            "Draft one concise product campaign using only the supplied facts. "
            "Do not invent discounts, prices, endorsements, certifications or warranties. "
            "Return one JSON object matching this schema: " + json.dumps(Campaign.model_json_schema())
        )}, {"role": "user", "content": brief.model_dump_json()}],
    )
    raw = response.choices[0].message.content
    if not raw or not evaluate("is_json", engine="local", output=raw).passed:
        raise ValueError("Model returned no usable JSON; revise the brief or retry.")
    return Campaign.model_validate_json(raw)

campaign = generate(brief)
print(campaign.headline, campaign.tagline, sep="\n")
print("Email subject:", campaign.email_subject)
print("Email draft:", campaign.email_body)
print("Social ad:", campaign.social_ad)

## Compose and export a campaign card

The first two brand colors become accents. Keep the headline dark on a light background so it stays readable even if the palette changes. Review product claims, copy, and layout before using the draft.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from textwrap import fill
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display


def make_card(brief, campaign):
    card = Image.new("RGB", (960, 540), "#FAF9F5")
    draw = ImageDraw.Draw(card)
    draw.rectangle((0, 0, 24, 540), fill=campaign.palette[0])
    draw.text((62, 50), brief.brand, fill="#202421", font=ImageFont.load_default(size=26))
    draw.line((62, 108, 890, 108), fill=campaign.palette[1], width=4)
    # Fit the full headline into the allotted area; do not silently clip long copy.
    for size in range(56, 13, -2):
        face = ImageFont.load_default(size=size)
        text = fill(campaign.headline, width=max(12, 1400 // size))
        box = draw.multiline_textbbox((0, 0), text, font=face, spacing=8)
        if box[2] <= 825 and box[3] <= 330:
            break
    else:
        raise ValueError("Shorten the headline for this card size.")
    draw.multiline_text((62, 140), text, fill="#202421", font=face, spacing=8)
    return card

folder = Path(mkdtemp(prefix="campaign-"))
card = make_card(brief, campaign)
(folder / "campaign.json").write_text(campaign.model_dump_json(indent=2), encoding="utf-8")
card.save(folder / "campaign-card.png")
assert Campaign.model_validate_json((folder / "campaign.json").read_text()) == campaign
with Image.open(folder / "campaign-card.png") as saved:
    assert saved.size == (960, 540)
print("Draft assets:", folder)
display(card)

## Use the same generator in a local Gradio interface

Change the facts and audience, then submit to generate another campaign. Each submission makes one model call. The example binds only to the loopback address. In a hosted notebook, use the cells above instead of exposing the interface or your model quota through a public share link.

In [ ]:
import os
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
import gradio as gr

def generate_for_form(brand, product, audience, facts, objective):
    current = Brief(brand=brand, product=product, audience=audience, facts=facts, objective=objective)
    draft = generate(current)
    return draft.model_dump(), make_card(current, draft)

interface = gr.Interface(fn=generate_for_form,
    inputs=[gr.Textbox(label=name.replace("_", " ").title(), value=value)
            for name, value in brief.model_dump().items()],
    outputs=[gr.JSON(label="Campaign"), gr.Image(label="Layout preview")],
    title="Brand Campaign Draft", analytics_enabled=False, flagging_mode="never")
if "google.colab" not in sys.modules:
    from urllib.request import urlopen
    _, local_url, _ = interface.launch(server_name="127.0.0.1", share=False,
                                        prevent_thread_lock=True, inline=False, quiet=True)
    with urlopen(local_url, timeout=10) as page:
        assert page.status == 200
    print("Local interface:", local_url)
else:
    print("Use generate(brief) and make_card(brief, campaign) in this notebook.")

Run `interface.close()` when you finish using the local interface. Re-run the brief and export cells for a second product; each export uses a fresh directory.